# **Turnover, costs, and capacity**

**Mechanics:** The live book trades at month-end, when SPY’s 200-day rule turns cash on or off, and when a name fails AAOIFI mid-month. Each trade has a cost. At a large enough fund size, a trade is a large share of that stock’s daily volume. This notebook measures both.

**White-paper focus:** At what AUM do costs or liquidity stop being compatible with steady-growth economics, before a Phase 2 capital raise.

**Frozen from 07–11:** FCF quality, 10% name cap, SMA → cash, `breach_exit="next_open"`, purification on the ex-date (about 1 bp of CAGR; costs here are separate).

**Definitions**

| Term | Meaning here |
| --- | --- |
| **Traded NAV** | Sum of \|weight changes\| that day (buys plus sells). Going to cash = 100%. |
| **One-way turnover** | Half of traded NAV. |
| **Flat cost** | `traded NAV × bps`. Same bps on every name. |
| **Participation** | Dollars we trade in a name ÷ that name’s 20-day median dollar volume. |
| **Capacity** | Largest AUM where no trade is more than **10% of ADV**. |

**Questions**

1. How much does this book actually trade per year, including SMA cash switches?
2. Do 5 / 10 / 20 / 50 bp flat costs change 2022 or the −9% drawdown?
3. At $50M, $100M, $250M, $500M, $1B, $2B, which sizes stay under 10% of ADV?

**Kill-style reading:** a cost assumption fails if 2022 is more than 5 points worse than SPUS, or train max drawdown is worse than SPUS.


## Setup


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import display


def resolve_notebook_dir() -> Path:
    candidates = [Path.cwd()]
    candidates.append(
        Path.home() / "Documents/Development/Monterey-Finance/Research/papers/12-turnover-capacity"
    )
    for path in candidates:
        try:
            resolved = path.resolve()
        except OSError:
            continue
        if resolved.is_dir() and (resolved / "code.ipynb").exists():
            return resolved
    return Path.cwd()


NB_DIR = resolve_notebook_dir()
os.chdir(NB_DIR)

REPO_ROOT = NB_DIR.parents[2]
RESEARCH_ROOT = NB_DIR.parents[1]
PAPER07 = RESEARCH_ROOT / "papers" / "07-book-construction"
HQ_ROOT = REPO_ROOT.parent / "halalquant"
PIP_DEPS = ["matplotlib", "pyarrow", "duckdb"]

if HQ_ROOT.is_dir() and (HQ_ROOT / "pyproject.toml").exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{HQ_ROOT}[cache]"],
        cwd=str(NB_DIR),
        check=False,
    )
    if str(HQ_ROOT) not in sys.path:
        sys.path.insert(0, str(HQ_ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PIP_DEPS, cwd=str(NB_DIR), check=False)
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

import importlib
import sleeves
import sleeves.backtest
import sleeves.book
import sleeves.costs
import sleeves.exits
import sleeves.lab

importlib.reload(sleeves.costs)
importlib.reload(sleeves.exits)
importlib.reload(sleeves.backtest)
importlib.reload(sleeves.book)
importlib.reload(sleeves.lab)
importlib.reload(sleeves)

os.environ.setdefault(
    "HALALQUANT_SEC_UA",
    "Monterey Finance Research halalquant/0.1.0 research@montereyfinance.com",
)

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sleeves import FrozenRules, Lab, calc_performance_stats
from sleeves.costs import (
    annualized_one_way,
    apply_flat_costs,
    apply_impact_costs,
    attach_adv,
    capacity_table,
    dollar_volume_panel,
    name_trades,
    trade_calendar,
)
from sleeves.exits import apply_breach_exits, detect_breaches
from sleeves.prices import spy_sma_risk_on
from sleeves.purify import weights_from_log

warnings.filterwarnings("ignore", category=FutureWarning)

START = "2019-12-19"
END = "2024-12-31"
TRAIN_START, TRAIN_END = "2020-01-01", "2022-12-31"
TEST_START, TEST_END = "2023-01-01", "2024-12-31"
RISK_FREE_RATE = 0.02
NAME_CAP = 0.10
ENGINE = {"fcf_quality": 1.0}
COST_LADDER = (0, 5, 10, 20, 50)
AUM_LIST = [50e6, 100e6, 250e6, 500e6, 1e9, 2e9]
MAX_PART = 0.10
KILL_2022_GAP = 0.05

CACHE_DIR = PAPER07 / "cache"
FIG_DIR = NB_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"notebook dir: {NB_DIR}")
print("engine: FCF 100%  name_cap=10%  SMA-to-cash  next_open exits")


## Step 1 — Locked book and the trade calendar


In [ ]:
metrics = pd.read_parquet(CACHE_DIR / "metrics.parquet")
prices = pd.read_parquet(CACHE_DIR / "prices.parquet")
prices["date"] = pd.to_datetime(prices["date"])
print(f"metrics {metrics.shape}  prices {prices.shape}  volume na {prices['volume'].isna().mean():.1%}")

rules = FrozenRules().with_book(
    sleeve_weights=ENGINE,
    name_cap=NAME_CAP,
    throttle="spy_sma",
    collapse_share_classes=True,
    cost_bps=10,
    risk_free_rate=RISK_FREE_RATE,
    rebalance_freq="ME",
    breach_exit="next_open",
    breach_monitor="filings",
    purify_schedule="ex_date",
)
lab = Lab.from_frames(metrics, prices, rules=rules, start=START, end=END)
print("Running locked book …", flush=True)
base_ret, base_log = lab.book().run(start=START)

def to_series(frame):
    s = frame.dropna(subset=["return"]).set_index("date")["return"].astype(float)
    s.index = pd.to_datetime(s.index)
    return s.sort_index()

gross = to_series(base_ret)
panel = lab.price_panel()
spy_px = lab.spy_series()
window = lab.rules.dual_momentum.sma_window
throttle_on = pd.Series(
    [spy_sma_risk_on(spy_px, dt, window) for dt in panel.index],
    index=pd.to_datetime(panel.index),
)
events = detect_breaches(
    metrics, base_log, panel, rules=rules, monitor="filings", end=END, throttle_on=throttle_on
)
weights_by_date = apply_breach_exits(
    weights_from_log(base_log), events, panel.index, lag="next_open", name_cap=NAME_CAP
)

print("Building trade calendar …", flush=True)
trades = trade_calendar(weights_by_date, panel.index, throttle_on=throttle_on, start=START)
ann = annualized_one_way(trades, START, END)
print(f"trade days {len(trades)}  one-way / year {ann:.1%}  traded NAV / year {2*ann:.1%}")
print(f"SMA to-cash days {int(trades['to_cash'].sum())}  from-cash days {int(trades['from_cash'].sum())}")
display(trades.sort_values("traded_nav", ascending=False).head(10))
trades.to_csv(NB_DIR / "trade_calendar.csv", index=False)

bench_px = (
    prices[prices["symbol"].isin(["SPY", "SPUS"])]
    .pivot(index="date", columns="symbol", values="adj_close")
    .sort_index()
    .ffill()
)
bench_ret = bench_px.pct_change(fill_method=None)
spy = bench_ret["SPY"].dropna()
spus = bench_ret["SPUS"].dropna()
spy = spy[(spy.index >= pd.Timestamp(START)) & (spy.index <= pd.Timestamp(END))]
spus = spus[(spus.index >= pd.Timestamp(START)) & (spus.index <= pd.Timestamp(END))]


## Step 2 — Flat cost ladder

Subtract `traded NAV × bps` on the day we trade. 10 bp is the stub used in papers 07–11.


In [ ]:
def sl(s, a, b):
    return s[(s.index >= pd.Timestamp(a)) & (s.index <= pd.Timestamp(b))].dropna()


def calendar_return(s, year):
    x = sl(s, f"{year}-01-01", f"{year}-12-31")
    return float((1 + x).prod() - 1) if not x.empty else np.nan


def score(s, label, cost_bps):
    full = sl(s, START, END)
    train = sl(s, TRAIN_START, TRAIN_END)
    test = sl(s, TEST_START, TEST_END)
    st = calc_performance_stats(full, label, rf=RISK_FREE_RATE)
    st_tr = calc_performance_stats(train, label, rf=RISK_FREE_RATE)
    st_te = calc_performance_stats(test, label, rf=RISK_FREE_RATE)
    return {
        "book": label,
        "cost_bps": cost_bps,
        "ret_2020": calendar_return(s, 2020),
        "ret_2022": calendar_return(s, 2022),
        "ret_2023": calendar_return(s, 2023),
        "ret_2024": calendar_return(s, 2024),
        "full_cagr": st["CAGR"],
        "full_vol": st["Volatility"],
        "full_sharpe": st["Sharpe"],
        "full_max_dd": st["Max Drawdown"],
        "full_calmar": st["Calmar"],
        "train_max_dd": st_tr["Max Drawdown"],
        "test_cagr": st_te["CAGR"],
    }

series_map = {}
rows = []
for bps in COST_LADDER:
    label = "Gross, 0 bp" if bps == 0 else f"Flat {bps:.0f} bp"
    print("Costing", label, flush=True)
    s = apply_flat_costs(gross, trades, bps)
    series_map[label] = s
    rows.append(score(s, label, bps))

for name, s in (("S&P 500", spy), ("Halal (SPUS)", spus)):
    series_map[name] = s
    rows.append(score(s, name, np.nan))

table = pd.DataFrame(rows)
spus_row = table.loc[table["book"] == "Halal (SPUS)"].iloc[0]


def kill_reason(row):
    if row["book"] in {"S&P 500", "Halal (SPUS)"}:
        return ""
    reasons = []
    if pd.notna(row["ret_2022"]) and row["ret_2022"] < spus_row["ret_2022"] - KILL_2022_GAP:
        reasons.append("2022 >5ppt worse than SPUS")
    if pd.notna(row["train_max_dd"]) and row["train_max_dd"] < spus_row["train_max_dd"]:
        reasons.append("train max DD worse than SPUS")
    return "; ".join(reasons)


table["kill"] = table.apply(kill_reason, axis=1)
table["pass"] = table["kill"] == ""
show = table.copy()
for c in ("ret_2020", "ret_2022", "ret_2023", "ret_2024", "full_cagr", "full_vol", "full_max_dd", "train_max_dd", "test_cagr"):
    show[c] = show[c].map(lambda x: "" if pd.isna(x) else f"{x:.2%}")
for c in ("full_sharpe", "full_calmar"):
    show[c] = show[c].map(lambda x: "" if pd.isna(x) else f"{x:.2f}")
display(show)
table.to_csv(NB_DIR / "cost_ladder.csv", index=False)


## Step 3 — Paths


In [ ]:
plot_keys = [k for k in ["Gross, 0 bp", "Flat 10 bp", "Flat 20 bp", "Flat 50 bp", "Halal (SPUS)"] if k in series_map]
fig, ax = plt.subplots(figsize=(11, 5))
for k in plot_keys:
    eq = (1 + sl(series_map[k], START, END)).cumprod()
    ax.plot(eq.index, eq.values, label=k)
ax.set_title("Growth of $1 — flat trading costs")
ax.set_ylabel("Growth of $1")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "equity-curves.png", dpi=140)
plt.show()

fig, ax = plt.subplots(figsize=(11, 4))
t = trades.copy()
t["date"] = pd.to_datetime(t["date"])
ax.bar(t["date"], t["one_way"], width=3)
ax.set_title("One-way turnover by trade date")
ax.set_ylabel("One-way")
fig.tight_layout()
fig.savefig(FIG_DIR / "turnover.png", dpi=140)
plt.show()


## Step 4 — ADV participation and AUM capacity

20-day median dollar volume (shares × close). A trade is too large if it is more than 10% of that number. Book capacity is the smallest AUM at which any name-day hits that line.


In [ ]:
print("Name-level trades + ADV …", flush=True)
nt = name_trades(weights_by_date, panel.index, throttle_on=throttle_on, start=START)
dv = dollar_volume_panel(prices)
nt = attach_adv(nt, dv, window=20)
print(f"name-trades {len(nt):,}  with ADV {int(nt['adv'].notna().sum()):,}")
nt.to_csv(NB_DIR / "name_trades.csv", index=False)

cap = capacity_table(nt, AUM_LIST, max_participation=MAX_PART)
cap["aum_label"] = cap["aum"].map(lambda x: f"${x/1e6:.0f}M" if x < 1e9 else f"${x/1e9:.1f}B")
display(cap[["aum_label", "max_participation", "median_participation", "n_over_10pct_adv", "frac_over_10pct_adv", "book_capacity", "ok"]])
cap.to_csv(NB_DIR / "capacity.csv", index=False)

book_cap = float(cap["book_capacity"].iloc[0]) if not cap.empty else np.nan
print(f"Book capacity at 10% ADV: ${book_cap:,.0f}" if pd.notna(book_cap) else "no capacity")

# Tightest names at $250M
nt250 = nt.dropna(subset=["adv"]).copy()
nt250["part"] = nt250["delta_w"].abs() * 250e6 / nt250["adv"]
display(nt250.sort_values("part", ascending=False).head(10)[["date", "symbol", "delta_w", "adv", "part"]])

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(cap["aum"] / 1e6, cap["max_participation"] * 100, marker="o")
ax.axhline(10, color="tab:red", linestyle="--", label="10% ADV")
ax.set_xlabel("AUM ($ millions)")
ax.set_ylabel("Max participation (%)")
ax.set_title("Largest ADV share vs fund size")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "capacity.png", dpi=140)
plt.show()


## Step 5 — Square-root impact at selected AUM

Base 5 bp plus `25 × sqrt(participation)` on each name’s traded weight. This is a conservative large-cap add-on, not a calibrated market-impact model.


In [ ]:
impact_rows = []
for aum in (100e6, 250e6, 500e6, 1e9):
    s = apply_impact_costs(gross, nt, aum=aum, base_bps=5.0, k=25.0)
    label = f"Impact @ ${aum/1e6:.0f}M" if aum < 1e9 else "Impact @ $1.0B"
    series_map[label] = s
    row = score(s, label, np.nan)
    row["aum"] = aum
    impact_rows.append(row)
impact = pd.DataFrame(impact_rows)
display(impact[["book", "full_cagr", "full_max_dd", "full_calmar", "ret_2022"]])
impact.to_csv(NB_DIR / "impact_by_aum.csv", index=False)


## Step 6 — Pick

Keep 10 bp as the reporting stub if it still passes 2022. Capacity is the 10% ADV AUM, not a return number. Do not raise past that size without stretching trades over more days.


In [ ]:
print("Flat cost ladder:")
print(table[table["cost_bps"].notna()][["book", "cost_bps", "full_cagr", "full_max_dd", "ret_2022", "pass", "kill"]].to_string(index=False))
print()
print("Capacity:")
print(cap[["aum_label", "max_participation", "n_over_10pct_adv", "ok"]].to_string(index=False))

ten = table.loc[table["cost_bps"] == 10].iloc[0]
if bool(ten["pass"]):
    cost_pick = 10.0
    cost_why = "10 bp still passes 2022 / train DD vs SPUS; keep as the reporting stub"
else:
    cost_pick = 5.0
    cost_why = "10 bp failed a kill bar; drop the stub to 5 bp"

ok_rows = cap[cap["ok"]]
if ok_rows.empty:
    cap_pick = float(cap["book_capacity"].iloc[0])
    cap_why = "no listed AUM stays under 10% ADV; use the binding name-day AUM"
else:
    cap_pick = float(ok_rows["aum"].max())
    cap_why = "largest listed AUM with every name-day ≤ 10% of 20-day median dollar volume"

print()
print(f"PICK cost_bps={cost_pick:.0f}  ({cost_why})")
print(f"PICK capacity=${cap_pick:,.0f}  ({cap_why})")
print(f"Binding 10% ADV AUM=${book_cap:,.0f}")
print()
print("Live rule still:")
print("  FrozenRules().with_book(")
print("      sleeve_weights={'fcf_quality': 1.0},")
print("      name_cap=0.10,")
print("      throttle='spy_sma',")
print("      breach_exit='next_open',")
print("      purify_schedule='ex_date',")
print(f"      cost_bps={cost_pick:.0f},")
print("  )")


## How to read the pick

- SMA cash switches are the large turnover days. Monthly FCF rebalance is smaller.
- Flat 10 bp is a reporting number, not a broker quote. 50 bp is a stress.
- 10% of ADV is an ops limit, not a law. Crossing it means you need more days to complete the trade, or a smaller book.
- Purification (~1 bp CAGR) is already decided in paper 11. Add it on top of these cost numbers; do not mix the two in one line.
- Phase 2 capital should not exceed the 10% ADV AUM unless the trade calendar is changed (multi-day cash switches).
